In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [9]:
!pip install sentence-transformers bert-score --quiet

import pandas as pd, numpy as np, torch, ast, os, json
from sentence_transformers import SentenceTransformer, InputExample, CrossEncoder
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from bert_score import score as bert_score_compute
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f'GPU disponibil: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memorie GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

GPU disponibil: True
Device: Tesla T4
Memorie GPU: 14.6 GB


In [10]:
INPUT_PATH  = '/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv'
OUTPUT_PATH = '/kaggle/working/'

df = pd.read_csv(INPUT_PATH)
print(f'Filme incarcate: {len(df):,}')
print(f'Coloane: {df.columns.tolist()}')

df['overview']         = df['overview'].fillna('').astype(str).str.strip()
df['tagline']          = df['tagline'].fillna('').astype(str).str.strip()
df['review_summary']   = df['review_summary'].fillna('').astype(str).str.strip().replace('nan', '')
df['overview_summary'] = df['overview_summary'].fillna('').astype(str).str.strip().replace('nan', '')
df['genres']           = df['genres'].fillna('').astype(str).str.strip()
df['keywords']         = df['keywords'].fillna('').astype(str).str.strip()
df['cast']             = df['cast'].fillna('').astype(str).str.strip()
df['director']         = df['director'].fillna('').astype(str).str.strip()

print()
for col in ['overview', 'overview_summary', 'review_summary', 'tagline']:
    n   = (df[col] != '').sum()
    avg = df[col].str.split().str.len().mean()
    print(f'  {col:<22} acoperire: {n:>6,} ({n/len(df)*100:.1f}%)   medie: ~{avg:.0f} cuvinte')

Filme incarcate: 40,197
Coloane: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary', 'input_review', 'review_summary']

  overview               acoperire: 40,197 (100.0%)   medie: ~47 cuvinte
  overview_summary       acoperire: 40,197 (100.0%)   medie: ~26 cuvinte
  review_summary         acoperire: 11,644 (29.0%)   medie: ~11 cuvinte
  tagline                acoperire: 40,197 (100.0%)   medie: ~9 cuvinte


In [11]:
def extract_cast_names(cast_raw, max_actors=4):
    if not isinstance(cast_raw, str) or not cast_raw.strip():
        return ''
    try:
        actors = ast.literal_eval(cast_raw)
        if isinstance(actors, list):
            names = [a['name'] for a in actors[:max_actors]
                     if isinstance(a, dict) and a.get('name')]
            return ', '.join(names)
    except Exception:
        pass
    return ''


def extract_keywords(kw_raw, max_kw=12):
    if not isinstance(kw_raw, str) or not kw_raw.strip():
        return ''
    try:
        parsed = ast.literal_eval(kw_raw)
        if isinstance(parsed, list):
            return ', '.join(str(k) for k in parsed[:max_kw])
    except Exception:
        pass
    return ', '.join(w.strip() for w in kw_raw.split(',')[:max_kw])


def parse_genres(genres_raw):
    if not isinstance(genres_raw, str) or not genres_raw.strip():
        return []
    try:
        parsed = ast.literal_eval(genres_raw)
        if isinstance(parsed, list):
            return [str(g).lower().strip() for g in parsed if g]
    except Exception:
        pass
    return [g.strip().lower() for g in genres_raw.split(',') if g.strip()]


print('Functii ajutatoare definite.')

Functii ajutatoare definite.


In [14]:
def clean_overview_summary(title, ovs):
    t = str(title).strip()
    s = str(ovs).strip()
    if len(t) > 3 and s.lower().startswith(t.lower()):
        s = s[len(t):].lstrip('. \n').strip()
    return s if s else str(ovs).strip()   


def build_doc_text_v4(row):
    parts = [str(row.get('title', '')).strip() + '.']

    ovs = clean_overview_summary(row.get('title', ''), row.get('overview_summary', ''))
    ov  = str(row.get('overview', '')).replace('nan', '').strip()
    plot = ovs if ovs else ov   
    if plot:
        parts.append('Plot: ' + plot)

    rev = str(row.get('review_summary', '')).replace('nan', '').strip()
    if rev:
        parts.append('Critics: ' + rev)

    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres:
        parts.append('Genres: ' + genres)

    cast = extract_cast_names(row.get('cast', ''), 4)
    if cast:
        parts.append('Cast: ' + cast)

    return ' '.join(parts)


def build_doc_text_no_review(row):
    parts = [str(row.get('title', '')).strip() + '.']

    ovs = clean_overview_summary(row.get('title', ''), row.get('overview_summary', ''))
    ov  = str(row.get('overview', '')).replace('nan', '').strip()
    plot = ovs if ovs else ov
    if plot:
        parts.append('Plot: ' + plot)

    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres:
        parts.append('Genres: ' + genres)

    cast = extract_cast_names(row.get('cast', ''), 4)
    if cast:
        parts.append('Cast: ' + cast)

    return ' '.join(parts)


df['doc_text']           = df.apply(build_doc_text_v4, axis=1)
df['doc_text_no_review'] = df.apply(build_doc_text_no_review, axis=1)

mask     = (df['tagline'] != '') & (df['doc_text'] != '')
df_valid = df[mask].copy().reset_index(drop=True)

print(f'Filme valide: {len(df_valid):,}')

title_dup = df_valid.apply(
    lambda r: str(r['doc_text']).count(str(r['title']).strip()) > 1, axis=1
).sum()
print(f'Titlu duplicat in doc_text: {title_dup} cazuri (tinta: 0)')

real_leak = df_valid.apply(
    lambda r: len(str(r['tagline'])) > 15
              and str(r['tagline']).lower() in str(r['doc_text']).lower(),
    axis=1
).sum()
print(f'Leakage (tagline complet in doc_text): {real_leak} cazuri')
print(f'  => {real_leak} filme TMDB au tagline incorporat in overview (~0.5%)')
print(f'  => Impact neglijabil la training (0.5% din {len(df_valid):,} exemple)')

ex_cu   = df_valid[df_valid['review_summary'] != ''].iloc[0]
ex_fara = df_valid[df_valid['review_summary'] == ''].iloc[0]
print(f'\nCU review: {ex_cu["title"]}')
print(ex_cu['doc_text'])
print(f'\nFARA review: {ex_fara["title"]}')
print(ex_fara['doc_text'])

Filme valide: 40,197
Titlu duplicat in doc_text: 4982 cazuri (tinta: 0)
Leakage (tagline complet in doc_text): 127 cazuri
  => 127 filme TMDB au tagline incorporat in overview (~0.5%)
  => Impact neglijabil la training (0.5% din 40,197 exemple)

CU review: Toy Story
Toy Story. Plot: Andys toys live happily in his room until Andys birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andys heart, Woody plots against Buzz. Critics: Tom Hanks leads a strong cast. Tim Allen is great, too, as Buzz Lightyear. Don Rickles (Mr. Potato Head), Wallace Shawn (Rex) and John Ratzenberger (Hamm) also bring fun. Genres: ['Family', 'Comedy', 'Animation', 'Adventure'] Cast: Tom Hanks, Tim Allen, Don Rickles, Jim Varney

FARA review: Grumpier Old Men
Grumpier Old Men. Plot: A family wedding reignites the ancient feud between nextdoor neighbors and fishing buddies John and Max. Meanwhile, a sultry Italian divorcée opens a Genres: ['Romance', 'Comedy'] Cast: Walter Matthau, Jack Lem

In [15]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')

sample = df_valid['doc_text'].sample(1000, random_state=42)
lengths = [len(tok(t, truncation=False)['input_ids']) for t in sample]

with_rev  = df_valid[df_valid['review_summary'] != '']['doc_text'].sample(min(500, (df_valid['review_summary'] != '').sum()), random_state=42)
no_rev    = df_valid[df_valid['review_summary'] == '']['doc_text'].sample(min(500, (df_valid['review_summary'] == '').sum()), random_state=42)

len_with = [len(tok(t, truncation=False)['input_ids']) for t in with_rev]
len_no   = [len(tok(t, truncation=False)['input_ids']) for t in no_rev]

print(f'doc_text CU review   — Medie: {np.mean(len_with):.0f} | Max: {max(len_with)} | >512: {sum(l>512 for l in len_with)}/{len(len_with)}')
print(f'doc_text FARA review — Medie: {np.mean(len_no):.0f}   | Max: {max(len_no)}   | >512: {sum(l>512 for l in len_no)}/{len(len_no)}')
print(f'Global               — Medie: {np.mean(lengths):.0f}  | 99p: {np.percentile(lengths,99):.0f}    | >512: {sum(l>512 for l in lengths)}/{len(lengths)}')

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

doc_text CU review   — Medie: 122 | Max: 158 | >512: 0/500
doc_text FARA review — Medie: 67   | Max: 97   | >512: 0/500
Global               — Medie: 83  | 99p: 139    | >512: 0/1000


In [16]:
BGE_QUERY_PREFIX = 'Represent this sentence: '

train_df, eval_df = train_test_split(df_valid, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
eval_df  = eval_df.reset_index(drop=True)
print(f'Training: {len(train_df):,} filme')
print(f'Eval:     {len(eval_df):,} filme')

train_examples = [
    InputExample(texts=[BGE_QUERY_PREFIX + row['tagline'], row['doc_text']])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

train_examples_no_review = [
    InputExample(texts=[BGE_QUERY_PREFIX + row['tagline'], row['doc_text_no_review']])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

n_with = sum(1 for _, r in train_df.iterrows() if r['review_summary'] not in ('', 'nan'))
print(f'\nExemple principale:  {len(train_examples):,}')
print(f'  - cu review:       {n_with:,} ({n_with/len(train_df)*100:.1f}%)')
print(f'  - fara review:     {len(train_df)-n_with:,}')
print(f'\nExemplu pereche:')
ex = train_examples[0]
print(f'  Query: {ex.texts[0]}')
print(f'  Doc:   {ex.texts[1][:200]}...')

Training: 36,177 filme
Eval:     4,020 filme

Exemple principale:  35,788
  - cu review:       10,438 (28.9%)
  - fara review:     25,739

Exemplu pereche:
  Query: Represent this sentence: She loved and trusted him until he cut off her head.
  Doc:   Savage Intruder. Plot: An enigmatic young man manipulates his way into working at the decaying mansion of a once prolific, but now reclusive and alcoholic, movie star named Katharine Genres: ['Thrille...


In [17]:
MODEL_NAME    = 'BAAI/bge-base-en-v1.5'
OUTPUT_V4A    = OUTPUT_PATH + 'sbert_v4a'  
OUTPUT_V4B    = OUTPUT_PATH + 'sbert_v4b'  
device        = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

import gc
torch.cuda.empty_cache()
gc.collect()

sbert = SentenceTransformer(MODEL_NAME, device=str(device))

print(f'Model: {MODEL_NAME}')
print(f'Device: {device}')
print(f'Embedding dim: {sbert.encode(["test"]).shape[1]}')

total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
free_mem  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
print(f'GPU: {total_mem:.1f} GB total | {free_mem:.1f} GB libera')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: BAAI/bge-base-en-v1.5
Device: cuda:0
Embedding dim: 768
GPU: 14.6 GB total | 14.1 GB libera


In [18]:
sbert = sbert.to(device)
_tok  = sbert.tokenizer

def tokenize_batch(texts, max_length=256):
    return _tok(texts, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt')

def get_embeddings(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert(encoded)
    emb = out['sentence_embedding']
    return F.normalize(emb, p=2, dim=1)

def mnr_loss_fn(emb_a, emb_p, scale=50.0):
    scores = torch.mm(emb_a, emb_p.T) * scale
    labels = torch.arange(len(emb_a), device=device)
    return F.cross_entropy(scores, labels)

def simple_collate(examples):
    return [e.texts[0] for e in examples], [e.texts[1] for e in examples]

BATCH_SIZE   = 128
EPOCHS       = 3
ACCUM_STEPS  = 1

train_dl    = DataLoader(train_examples, batch_size=BATCH_SIZE,
                         shuffle=True, collate_fn=simple_collate)
total_steps  = len(train_dl) * EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(sbert.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler    = GradScaler('cuda')

print(f'Antrenare V4a:')
print(f'  model={MODEL_NAME}')
print(f'  batch={BATCH_SIZE} | negatives/query={BATCH_SIZE-1} | epoci={EPOCHS}')
print(f'  total steps={total_steps:,} | warmup={warmup_steps:,}')

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

for epoch in range(EPOCHS):
    sbert.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_dl), total=len(train_dl),
                desc=f'Epoch {epoch+1}/{EPOCHS}')
    for step, (anchors, positives) in pbar:
        enc_a = tokenize_batch(anchors)
        enc_p = tokenize_batch(positives)

        with autocast('cuda'):
            emb_a = get_embeddings(enc_a)
            emb_p = get_embeddings(enc_p)
            loss  = mnr_loss_fn(emb_a, emb_p)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(sbert.parameters(), max_norm=1.0)
        scaler.step(optimizer)   
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(train_dl)
    print(f'Epoch {epoch+1}/{EPOCHS} — Loss mediu: {avg_loss:.4f}')

    ckpt = OUTPUT_PATH + f'sbert_v4a_epoch{epoch+1}'
    sbert.save(ckpt)
    print(f'  Checkpoint: {ckpt}')

sbert.save(OUTPUT_V4A)
print(f'\nV4a salvat: {OUTPUT_V4A}')

Antrenare V4a:
  model=BAAI/bge-base-en-v1.5
  batch=128 | negatives/query=127 | epoci=3
  total steps=840 | warmup=84


Epoch 1/3:   0%|          | 0/280 [00:00<?, ?it/s]

Epoch 1/3 — Loss mediu: 2.7096


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Checkpoint: /kaggle/working/sbert_v4a_epoch1


...

Epoch 3/3 — Loss mediu: 2.0489


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Checkpoint: /kaggle/working/sbert_v4a_epoch3


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


V4a salvat: /kaggle/working/sbert_v4a


In [19]:
BATCH_EMB = 256

queries_prefixed = [BGE_QUERY_PREFIX + t for t in df_valid['tagline'].tolist()]
docs             = df_valid['doc_text'].tolist()
docs_no_review   = df_valid['doc_text_no_review'].tolist()

sbert_base = SentenceTransformer(MODEL_NAME, device=str(device))
sbert_v4a  = SentenceTransformer(OUTPUT_V4A,  device=str(device))

print('Generare embeddings Baseline BGE (fara fine-tuning)...')
q_emb_base = sbert_base.encode(queries_prefixed, batch_size=BATCH_EMB,
                                show_progress_bar=True, convert_to_numpy=True,
                                normalize_embeddings=True)
d_emb_base = sbert_base.encode(docs, batch_size=BATCH_EMB,
                                show_progress_bar=True, convert_to_numpy=True,
                                normalize_embeddings=True)

print('\nGenerare embeddings V4a (fine-tuned, cu review)...')
q_emb_v4a = sbert_v4a.encode(queries_prefixed, batch_size=BATCH_EMB,
                               show_progress_bar=True, convert_to_numpy=True,
                               normalize_embeddings=True)
d_emb_v4a = sbert_v4a.encode(docs, batch_size=BATCH_EMB,
                               show_progress_bar=True, convert_to_numpy=True,
                               normalize_embeddings=True)

print('\nGenerare embeddings V4a ablatie (fara review in doc)...')
d_emb_v4a_norev = sbert_v4a.encode(docs_no_review, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)

np.save(OUTPUT_PATH + 'q_emb_base.npy',       q_emb_base)
np.save(OUTPUT_PATH + 'd_emb_base.npy',       d_emb_base)
np.save(OUTPUT_PATH + 'q_emb_v4a.npy',        q_emb_v4a)
np.save(OUTPUT_PATH + 'd_emb_v4a.npy',        d_emb_v4a)
np.save(OUTPUT_PATH + 'd_emb_v4a_norev.npy',  d_emb_v4a_norev)
print('\nEmbeddings salvate!')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generare embeddings Baseline BGE (fara fine-tuning)...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Generare embeddings V4a ablatie (fara review in doc)...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Embeddings salvate!


In [20]:
df_valid['genres_list'] = df_valid['genres'].apply(parse_genres)

genre_to_indices = {}
for i, genres in enumerate(df_valid['genres_list']):
    for g in genres:
        genre_to_indices.setdefault(g, []).append(i)

print(f'Genuri unice: {len(genre_to_indices)}')


def evaluate_full_pool(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                       batch_size=512, label=''):
    N    = len(query_emb)
    hits = {k: 0 for k in k_values}
    max_k = max(k_values)

    for start in range(0, N, batch_size):
        end    = min(start + batch_size, N)
        scores = cosine_similarity(query_emb[start:end], doc_emb)
        top_k  = np.argpartition(scores, -max_k, axis=1)[:, -max_k:]

        for i, global_idx in enumerate(range(start, end)):
            top_sorted = top_k[i][np.argsort(scores[i][top_k[i]])[::-1]]
            for k in k_values:
                if global_idx in top_sorted[:k]:
                    hits[k] += 1

        if start % 10000 == 0 and start > 0:
            print(f'  {label} {start}/{N}...')

    return {k: hits[k] / N for k in k_values}


def evaluate_genre_filtered(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                             sample_n=3000, min_pool=30, label=''):
    np.random.seed(42)
    indices    = np.random.choice(len(query_emb), min(sample_n, len(query_emb)), replace=False)
    hits       = {k: 0 for k in k_values}
    pool_sizes = []

    for idx in indices:
        pool = set()
        for g in df_valid['genres_list'].iloc[idx]:
            pool.update(genre_to_indices.get(g, []))
        pool.add(idx)
        if len(pool) < min_pool:
            pool = set(range(len(df_valid)))

        pool_arr  = np.array(sorted(pool))
        pool_sizes.append(len(pool_arr))
        scores    = cosine_similarity(query_emb[idx:idx+1], doc_emb[pool_arr])[0]
        local_idx = int(np.where(pool_arr == idx)[0][0])
        ranked    = np.argsort(scores)[::-1]
        rank      = int(np.where(ranked == local_idx)[0][0]) + 1

        for k in k_values:
            if rank <= k:
                hits[k] += 1

    n = len(indices)
    print(f'  Pool mediu per query: {np.mean(pool_sizes):.0f} filme')
    return {k: hits[k] / n for k in k_values}


print('Functii de evaluare definite.')

Genuri unice: 19
Functii de evaluare definite.


In [21]:
results = {}

print('FULL POOL (toate cele 40k filme)')

print('\nBaseline BGE (fara fine-tuning):')
results['base_full'] = evaluate_full_pool(q_emb_base, d_emb_base, label='base')
print(f'  {results["base_full"]}')

print('\nV4a (BGE fine-tuned, batch=128, cu review):')
results['v4a_full'] = evaluate_full_pool(q_emb_v4a, d_emb_v4a, label='v4a')
print(f'  {results["v4a_full"]}')

print('\nV4a Ablatie (fara review in doc_text):')
results['v4a_norev_full'] = evaluate_full_pool(q_emb_v4a, d_emb_v4a_norev, label='v4a-norev')
print(f'  {results["v4a_norev_full"]}')

print('\nGENRE-FILTERED POOL')

print('\nBaseline BGE:')
results['base_genre'] = evaluate_genre_filtered(q_emb_base, d_emb_base)
print(f'  {results["base_genre"]}')

print('\nV4a fine-tuned:')
results['v4a_genre'] = evaluate_genre_filtered(q_emb_v4a, d_emb_v4a)
print(f'  {results["v4a_genre"]}')

with open(OUTPUT_PATH + 'results_v4a.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate V4a salvate!')

FULL POOL (toate cele 40k filme)

Baseline BGE (fara fine-tuning):
  {1: 0.05930790855038933, 3: 0.0883648033435331, 5: 0.10416200213946314, 10: 0.12677563002214096, 20: 0.15438963106699505}

V4a (BGE fine-tuned, batch=128, cu review):
  {1: 0.10411224718262557, 3: 0.1590914744881459, 5: 0.1914321964325696, 10: 0.2376794288130955, 20: 0.2941513048237431}

V4a Ablatie (fara review in doc_text):
  {1: 0.10396298231211284, 3: 0.1572505410851556, 5: 0.18735228997188844, 10: 0.23454486653232828, 20: 0.28783242530537106}

GENRE-FILTERED POOL

Baseline BGE:
  Pool mediu per query: 15835 filme
  {1: 0.08333333333333333, 3: 0.119, 5: 0.136, 10: 0.16366666666666665, 20: 0.198}

V4a fine-tuned:
  Pool mediu per query: 15835 filme
  {1: 0.12066666666666667, 3: 0.16766666666666666, 5: 0.196, 10: 0.23833333333333334, 20: 0.2773333333333333}

Rezultate V4a salvate!


In [22]:
HN_PER_QUERY = 5   
MINE_TOPK    = 60   
BATCH_MINE   = 512  

print(f'Minare hard negatives din {len(train_examples):,} exemple...')
print(f'  Top-{MINE_TOPK} candidati, selectam {HN_PER_QUERY} HN per query')
print(f'  Filtru false-negatives: skip daca >= 2 genuri comune\n')

train_taglines_prefixed = [
    BGE_QUERY_PREFIX + str(row['tagline'])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]
train_docs = [
    str(row['doc_text'])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]
train_genres_list = [
    parse_genres(str(row['genres']))
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

print('Encoding query embeddings pentru train set...')
q_emb_train = sbert_v4a.encode(
    train_taglines_prefixed, batch_size=256,
    show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True
)
print('Encoding doc embeddings pentru train set...')
d_emb_train = sbert_v4a.encode(
    train_docs, batch_size=256,
    show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True
)

hard_neg_examples = []
skipped = 0

for i in tqdm(range(len(train_taglines_prefixed)), desc='Mining HN'):
    scores  = cosine_similarity(q_emb_train[i:i+1], d_emb_train)[0]
    scores[i] = -1.0  
    top_idx = np.argsort(scores)[::-1][:MINE_TOPK]

    genres_q = set(train_genres_list[i])
    hard_negs = []

    for idx in top_idx:
        genres_c = set(train_genres_list[idx])
        if len(genres_q & genres_c) >= 2:
            skipped += 1
            continue
        hard_negs.append(train_docs[idx])
        if len(hard_negs) >= HN_PER_QUERY:
            break

    for hn in hard_negs:
        hard_neg_examples.append(
            InputExample(texts=[
                train_taglines_prefixed[i],  
                train_docs[i],             
                hn                         
            ])
        )

print(f'\nHard negative triplets generate: {len(hard_neg_examples):,}')
print(f'Candidati filtrati (false negatives posibile): {skipped:,}')
print(f'Exemplu triplet:')
ex = hard_neg_examples[0]
print(f'  Anchor:   {ex.texts[0]}')
print(f'  Pozitiv:  {ex.texts[1][:120]}...')
print(f'  HardNeg:  {ex.texts[2][:120]}...')

Minare hard negatives din 35,788 exemple...
  Top-60 candidati, selectam 5 HN per query
  Filtru false-negatives: skip daca >= 2 genuri comune

Encoding query embeddings pentru train set...


Batches:   0%|          | 0/140 [00:00<?, ?it/s]

Encoding doc embeddings pentru train set...


Batches:   0%|          | 0/140 [00:00<?, ?it/s]

Mining HN:   0%|          | 0/35788 [00:00<?, ?it/s]


Hard negative triplets generate: 178,683
Candidati filtrati (false negatives posibile): 108,329
Exemplu triplet:
  Anchor:   Represent this sentence: She loved and trusted him until he cut off her head.
  Pozitiv:  Savage Intruder. Plot: An enigmatic young man manipulates his way into working at the decaying mansion of a once prolifi...
  HardNeg:  A Severed Head. Plot: Antonia LynchGibbon, wife of uppercrust wine dealer Martin, falls in love with her husbands best f...


In [23]:
torch.cuda.empty_cache(); gc.collect()
sbert_v4b = SentenceTransformer(OUTPUT_V4A, device=str(device))
sbert_v4b = sbert_v4b.to(device)
_tok_v4b  = sbert_v4b.tokenizer

def tokenize_v4b(texts, max_length=256):
    return _tok_v4b(texts, padding=True, truncation=True,
                   max_length=max_length, return_tensors='pt')

def get_emb_v4b(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert_v4b(encoded)
    return F.normalize(out['sentence_embedding'], p=2, dim=1)

def mnr_triplet_loss(emb_a, emb_p, emb_n, scale=50.0):
    emb_docs = torch.cat([emb_p, emb_n], dim=0)  
    scores   = torch.mm(emb_a, emb_docs.T) * scale 
    labels   = torch.arange(len(emb_a), device=device) 
    return F.cross_entropy(scores, labels)

def hn_collate(examples):
    anchors   = [e.texts[0] for e in examples]
    positives = [e.texts[1] for e in examples]
    negatives = [e.texts[2] for e in examples]
    return anchors, positives, negatives

BATCH_HN  = 64    
EPOCHS_HN = 2   

hn_dl       = DataLoader(hard_neg_examples, batch_size=BATCH_HN,
                         shuffle=True, collate_fn=hn_collate)
total_steps_hn  = len(hn_dl) * EPOCHS_HN
warmup_steps_hn = int(0.05 * total_steps_hn)

optimizer_hn = AdamW(sbert_v4b.parameters(), lr=1e-5, weight_decay=0.01)
scheduler_hn = get_linear_schedule_with_warmup(optimizer_hn, warmup_steps_hn, total_steps_hn)
scaler_hn    = GradScaler('cuda')

print(f'Fine-tuning V4b cu hard negatives:')
print(f'  Start din: {OUTPUT_V4A}')
print(f'  batch={BATCH_HN} | triplete={len(hard_neg_examples):,} | epoci={EPOCHS_HN}')

for epoch in range(EPOCHS_HN):
    sbert_v4b.train()
    total_loss = 0.0
    optimizer_hn.zero_grad()

    pbar = tqdm(enumerate(hn_dl), total=len(hn_dl),
                desc=f'V4b Epoch {epoch+1}/{EPOCHS_HN}')
    for step, (anchors, positives, negatives) in pbar:
        enc_a = tokenize_v4b(anchors)
        enc_p = tokenize_v4b(positives)
        enc_n = tokenize_v4b(negatives)

        with autocast('cuda'):
            emb_a = get_emb_v4b(enc_a)
            emb_p = get_emb_v4b(enc_p)
            emb_n = get_emb_v4b(enc_n)
            loss  = mnr_triplet_loss(emb_a, emb_p, emb_n)

        scaler_hn.scale(loss).backward()
        scaler_hn.unscale_(optimizer_hn)
        torch.nn.utils.clip_grad_norm_(sbert_v4b.parameters(), max_norm=1.0)
        scaler_hn.step(optimizer_hn)
        scaler_hn.update()
        scheduler_hn.step()
        optimizer_hn.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    print(f'V4b Epoch {epoch+1} — Loss mediu: {total_loss/len(hn_dl):.4f}')

sbert_v4b.save(OUTPUT_V4B)
print(f'\nV4b salvat: {OUTPUT_V4B}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuning V4b cu hard negatives:
  Start din: /kaggle/working/sbert_v4a
  batch=64 | triplete=178,683 | epoci=2


V4b Epoch 1/2:   0%|          | 0/2792 [00:00<?, ?it/s]

V4b Epoch 1 — Loss mediu: 2.3297


V4b Epoch 2/2:   0%|          | 0/2792 [00:00<?, ?it/s]

V4b Epoch 2 — Loss mediu: 1.7547


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


V4b salvat: /kaggle/working/sbert_v4b


In [24]:
sbert_v4b_eval = SentenceTransformer(OUTPUT_V4B, device=str(device))

print('Generare embeddings V4b...')
q_emb_v4b = sbert_v4b_eval.encode(queries_prefixed, batch_size=BATCH_EMB,
                                   show_progress_bar=True, convert_to_numpy=True,
                                   normalize_embeddings=True)
d_emb_v4b = sbert_v4b_eval.encode(docs, batch_size=BATCH_EMB,
                                   show_progress_bar=True, convert_to_numpy=True,
                                   normalize_embeddings=True)

np.save(OUTPUT_PATH + 'q_emb_v4b.npy', q_emb_v4b)
np.save(OUTPUT_PATH + 'd_emb_v4b.npy', d_emb_v4b)

print('\nEvaluare V4b full pool...')
results['v4b_full'] = evaluate_full_pool(q_emb_v4b, d_emb_v4b, label='v4b')
print(f'  {results["v4b_full"]}')

print('\nEvaluare V4b genre-filtered...')
results['v4b_genre'] = evaluate_genre_filtered(q_emb_v4b, d_emb_v4b)
print(f'  {results["v4b_genre"]}')

with open(OUTPUT_PATH + 'results_v4b.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate V4b salvate!')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generare embeddings V4b...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Evaluare V4b full pool...
  {1: 0.16439037739134763, 3: 0.2631042117570963, 5: 0.31288404607309, 10: 0.3871930741100082, 20: 0.466701495136453}

Evaluare V4b genre-filtered...
  Pool mediu per query: 15835 filme
  {1: 0.09966666666666667, 3: 0.143, 5: 0.16566666666666666, 10: 0.19766666666666666, 20: 0.24033333333333334}

Rezultate V4b salvate!


In [26]:
SAMPLE_BS  = 300
BATCH_BERT = 64

np.random.seed(42)
sample_idx = np.random.choice(len(df_valid), SAMPLE_BS, replace=False)

def get_top1_idx(q_emb, d_emb, idx):
    scores = cosine_similarity(q_emb[idx:idx+1], d_emb)[0]
    return int(np.argmax(scores))

def get_bert_text(idx):
    row = df_valid.iloc[idx]
    return clean_overview_summary(row['title'], row['overview_summary'])

refs      = [get_bert_text(i) for i in sample_idx]

hyps_base = [get_bert_text(get_top1_idx(q_emb_base, d_emb_base, i)) for i in sample_idx]
hyps_v4a  = [get_bert_text(get_top1_idx(q_emb_v4a,  d_emb_v4a,  i)) for i in sample_idx]
hyps_v4b  = [get_bert_text(get_top1_idx(q_emb_v4b,  d_emb_v4b,  i)) for i in sample_idx]

hit1_base = [1 if get_top1_idx(q_emb_base, d_emb_base, i) == i else 0 for i in sample_idx]
hit1_v4a  = [1 if get_top1_idx(q_emb_v4a,  d_emb_v4a,  i) == i else 0 for i in sample_idx]
hit1_v4b  = [1 if get_top1_idx(q_emb_v4b,  d_emb_v4b,  i) == i else 0 for i in sample_idx]

print(f'Sample: {SAMPLE_BS} filme')
print(f'Hit@1 — base: {sum(hit1_base)/len(hit1_base):.3f} | v4a: {sum(hit1_v4a)/len(hit1_v4a):.3f} | v4b: {sum(hit1_v4b)/len(hit1_v4b):.3f}')

bert_results = {}
for name, hyps in [('base', hyps_base), ('v4a', hyps_v4a), ('v4b', hyps_v4b)]:
    print(f'\nBERTScore {name}...')
    _, _, F1 = bert_score_compute(
        hyps, refs,
        lang='en',
        model_type='bert-base-uncased',
        batch_size=BATCH_BERT,
        verbose=False
    )
    bert_results[name] = F1.numpy()
    hits_list = {'base': hit1_base, 'v4a': hit1_v4a, 'v4b': hit1_v4b}[name]
    f1_miss = F1.numpy()[[h == 0 for h in hits_list]]
    print(f'  F1 global:  {F1.mean():.4f} ± {F1.std():.4f}')
    print(f'  F1 miss:    {f1_miss.mean():.4f}  (cand filmul exact nu e recuperat)')

print('\nBERTScore calculat!')

Sample: 300 filme
Hit@1 — base: 0.063 | v4a: 0.110 | v4b: 0.107

BERTScore base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1 global:  0.4624 ± 0.1480
  F1 miss:    0.4260  (cand filmul exact nu e recuperat)

BERTScore v4a...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1 global:  0.5037 ± 0.1808
  F1 miss:    0.4423  (cand filmul exact nu e recuperat)

BERTScore v4b...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1 global:  0.5050 ± 0.1769
  F1 miss:    0.4458  (cand filmul exact nu e recuperat)

BERTScore calculat!


In [ ]:
CE_MODEL = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
cross_encoder = CrossEncoder(CE_MODEL, device=str(device))
print(f'Cross-encoder incarcat: {CE_MODEL}')


def evaluate_with_reranking(query_emb, doc_emb, queries_raw, docs_raw,
                             k_values=[1, 3, 5, 10], sample_n=500,
                             retrieval_k=50):
    np.random.seed(42)
    indices = np.random.choice(len(query_emb), min(sample_n, len(query_emb)), replace=False)
    hits    = {k: 0 for k in k_values}

    for idx in tqdm(indices, desc='Reranking eval'):
        scores   = cosine_similarity(query_emb[idx:idx+1], doc_emb)[0]
        top50    = np.argsort(scores)[::-1][:retrieval_k]

        query_raw = queries_raw[idx].replace(BGE_QUERY_PREFIX, '')
        pairs     = [(query_raw, docs_raw[j]) for j in top50]
        ce_scores = cross_encoder.predict(pairs, batch_size=32)
        reranked  = top50[np.argsort(ce_scores)[::-1]]

        for k in k_values:
            if idx in reranked[:k]:
                hits[k] += 1

    n = len(indices)
    return {k: hits[k] / n for k in k_values}


print('\nEvaluare V4b + Cross-Encoder reranking (sample 500)...')
results['v4b_rerank'] = evaluate_with_reranking(
    q_emb_v4b, d_emb_v4b,
    queries_prefixed, docs,
    k_values=[1, 3, 5, 10]
)
print(f'  {results["v4b_rerank"]}')

with open(OUTPUT_PATH + 'results_final.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate finale salvate!')

In [ ]:
import pandas as pd

print(f'{"COMPARATIE MODELE — METRICI DUALE":^78}')
print(f'{"Model":<38} {"Pool":<7} {"Hit@1":>6} {"Hit@5":>6} {"Hit@10":>7} {"Hit@20":>7}')

rows_hk = [
    ('Baseline BGE     (no fine-tune)',   'full',  'base_full'),
    ('[Ref] V2 all-mpnet MNR',            'full',  None),
    ('[Ref] V3 all-mpnet + review',       'full',  None),
    ('V4a BGE  (batch=128, cu review)',   'full',  'v4a_full'),
    ('V4a Ablatie  (fara review)',        'full',  'v4a_norev_full'),
    ('V4b BGE  (+ hard negatives)',       'full',  'v4b_full'),
    ('Baseline BGE',                      'genre', 'base_genre'),
    ('V4a fine-tuned',                    'genre', 'v4a_genre'),
    ('V4b + hard negatives',             'genre', 'v4b_genre'),
]

ref_vals = {
    '[Ref] V2 all-mpnet MNR':        {1: 0.100, 5: 0.187, 10: 0.236, 20: 0.293},
    '[Ref] V3 all-mpnet + review':   {1: None,  5: None,  10: 0.237, 20: None},
}

for label, pool, key in rows_hk:
    if key and key in results:
        r = results[key]
        h1  = f"{r[1]*100:.1f}%"
        h5  = f"{r[5]*100:.1f}%"
        h10 = f"{r[10]*100:.1f}%"
        h20 = f"{r[20]*100:.1f}%"
    elif label in ref_vals:
        rv  = ref_vals[label]
        h1  = f"{rv[1]*100:.1f}%"  if rv[1]  else '   —'
        h5  = f"{rv[5]*100:.1f}%"  if rv[5]  else '   —'
        h10 = f"{rv[10]*100:.1f}%" if rv[10] else '   —'
        h20 = f"{rv[20]*100:.1f}%" if rv[20] else '   —'
    else:
        continue
    print(f'{label:<38} {pool:<7} {h1:>6} {h5:>6} {h10:>7} {h20:>7}')

print()
print('=' * 65)
print(f'{"BERTSCORE F1 (overview_summary — N=300)": ^65}')
print('=' * 65)
print(f'{"Model":<30} {"BS-F1 global":>13} {"BS-F1 miss (Hit@1=0)":>22}')
print('-' * 65)

for name, hits_list, label in [
    ('base', hit1_base, 'Baseline BGE'),
    ('v4a',  hit1_v4a,  'V4a fine-tuned'),
    ('v4b',  hit1_v4b,  'V4b + hard negatives'),
]:
    f1_all  = bert_results[name]
    f1_miss = bert_results[name][[h == 0 for h in hits_list]]
    print(f'{label:<30} {f1_all.mean():>13.4f} {f1_miss.mean():>22.4f}')

print('=' * 65)
print('  BS-F1 global  = similaritate semantica medie a top-1 recuperat vs. corect')
print('  BS-F1 miss    = BS-F1 doar cand filmul exact NU e recuperat (Hit@1=0)')
print('  Textul comparat: overview_summary (acoperire 97%, ~28 cuvinte/film)')

if 'v4b_rerank' in results:
    print()
    print('=' * 55)
    print(f'{"V4b + Cross-Encoder Reranking (top-50 → rerank)": ^55}')
    print('=' * 55)
    r = results['v4b_rerank']
    print(f'  Hit@1: {r[1]*100:.1f}%   Hit@3: {r[3]*100:.1f}%   Hit@5: {r[5]*100:.1f}%   Hit@10: {r[10]*100:.1f}%')
    print('=' * 55)

                      COMPARATIE MODELE — METRICI DUALE                       
Model                                  Pool     Hit@1  Hit@5  Hit@10  Hit@20
Baseline BGE     (no fine-tune)        full      5.9%  10.4%   12.7%   15.4%
[Ref] V2 all-mpnet MNR                 full     10.0%  18.7%   23.6%   29.3%
[Ref] V3 all-mpnet + review            full         —      —   23.7%       —
V4a BGE  (batch=128, cu review)        full     10.4%  19.1%   23.8%   29.4%
V4a Ablatie  (fara review)             full     10.4%  18.7%   23.5%   28.8%
V4b BGE  (+ hard negatives)            full     16.4%  31.3%   38.7%   46.7%
Baseline BGE                           genre     8.3%  13.6%   16.4%   19.8%
V4a fine-tuned                         genre    12.1%  19.6%   23.8%   27.7%
V4b + hard negatives                   genre    10.0%  16.6%   19.8%   24.0%

             BERTSCORE F1 (overview_summary — N=300)             
Model                           BS-F1 global   BS-F1 miss (Hit@1=0)
----------------

In [30]:
CE_MODEL = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
if 'cross_encoder' not in dir() or cross_encoder is None:
    print(f'Incarcare cross-encoder: {CE_MODEL}')
    cross_encoder = CrossEncoder(CE_MODEL, device=str(device))
    print('Cross-encoder gata.')

def recommend_v4(query: str, top_k: int = 5,
                  use_reranking: bool = True, retrieval_k: int = 50):
    q_prefixed = BGE_QUERY_PREFIX + query
    q_emb      = sbert_v4b_eval.encode([q_prefixed], convert_to_numpy=True,
                                        normalize_embeddings=True)
    bi_scores  = cosine_similarity(q_emb, d_emb_v4b)[0]

    if use_reranking:
        top_bi   = np.argsort(bi_scores)[::-1][:retrieval_k]
        pairs    = [(query, docs[i]) for i in top_bi]
        ce_sc    = cross_encoder.predict(pairs, batch_size=32)
        top_idx  = top_bi[np.argsort(ce_sc)[::-1]][:top_k]
        mode_str = f'V4b + Cross-Encoder (bi-top{retrieval_k} → rerank)'
    else:
        top_idx  = np.argsort(bi_scores)[::-1][:top_k]
        mode_str = 'V4b bi-encoder only'

    print(f'\nQuery: "{query}"')
    print(f'Mod:   {mode_str}')
    print(f'{"Rang":<5} {"Rev":>4} {"Titlu":<40} {"Gen":<28} {"Scor":>6}')
    print('-' * 88)

    for rank, idx in enumerate(top_idx, 1):
        row     = df_valid.iloc[idx]
        rev_ico = 'Yes' if row['review_summary'] not in ('', 'nan') else ' '
        genres  = str(row['genres'])[:26]
        score   = bi_scores[idx]
        print(f'{rank:<5} {rev_ico:>4} {str(row["title"]):<40} {genres:<28} {score:>6.3f}')


print('Legenda: Rev Yes = film are review_summary\n')

demo_queries = [
    'animated toys that come to life and go on adventures',
    'a superhero saves the world from an alien invasion',
    'romantic comedy set in New York with funny misunderstandings',
    'psychological thriller about dreams and reality',
    'a young wizard discovers his magical powers at school',
]

for q in demo_queries:
    recommend_v4(q, top_k=5, use_reranking=True)

Incarcare cross-encoder: cross-encoder/ms-marco-MiniLM-L-12-v2


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder gata.
Legenda: Rev Yes = film are review_summary


Query: "animated toys that come to life and go on adventures"
Mod:   V4b + Cross-Encoder (bi-top50 → rerank)
Rang   Rev Titlu                                    Gen                            Scor
----------------------------------------------------------------------------------------
1          The Adventures of Scamper the Penguin    ['Adventure', 'Animation',    0.706
2          Rudolph the Red-Nosed Reindeer & the Island of Misfit Toys ['Animation', 'Family']       0.724
3          The Brave Little Toaster                 ['Animation', 'Adventure',    0.698
4          Barbie: Star Light Adventure             ['Animation', 'Family', 'S    0.702
5          Raggedy Ann & Andy: A Musical Adventure! ['Animation', 'Adventure',    0.711

Query: "a superhero saves the world from an alien invasion"
Mod:   V4b + Cross-Encoder (bi-top50 → rerank)
Rang   Rev Titlu                                    Gen                            

In [1]:
import numpy as np
import pandas as pd
import ast
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv')
for col in ['overview', 'overview_summary', 'review_summary', 'tagline', 'genres', 'keywords', 'cast']:
    df[col] = df[col].fillna('').astype(str).str.strip().replace('nan', '')

def clean_overview_summary(title, ovs):
    t = str(title).strip()
    s = str(ovs).strip()
    if len(t) > 3 and s.lower().startswith(t.lower()):
        s = s[len(t):].lstrip('. \n').strip()
    return s if s else str(ovs).strip()

def extract_cast_names(cast_raw, max_actors=4):
    if not isinstance(cast_raw, str) or not cast_raw.strip():
        return ''
    try:
        actors = ast.literal_eval(cast_raw)
        if isinstance(actors, list):
            return ', '.join(a['name'] for a in actors[:max_actors] if isinstance(a, dict) and a.get('name'))
    except:
        pass
    return ''

def build_doc_text_v4(row):
    parts = [str(row.get('title', '')).strip() + '.']
    ovs = clean_overview_summary(row.get('title', ''), row.get('overview_summary', ''))
    ov  = str(row.get('overview', '')).replace('nan', '').strip()
    plot = ovs if ovs else ov
    if plot: parts.append('Plot: ' + plot)
    rev = str(row.get('review_summary', '')).replace('nan', '').strip()
    if rev: parts.append('Critics: ' + rev)
    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres: parts.append('Genres: ' + genres)
    cast = extract_cast_names(row.get('cast', ''), 4)
    if cast: parts.append('Cast: ' + cast)
    return ' '.join(parts)

df['doc_text'] = df.apply(build_doc_text_v4, axis=1)
df_valid = df[(df['tagline'] != '') & (df['doc_text'] != '')].copy().reset_index(drop=True)

_, eval_df = train_test_split(df_valid, test_size=0.1, random_state=42)
eval_df = eval_df.reset_index(drop=True)

q_emb = np.load('/kaggle/input/notebooks/catalinalupu/notebook-varianta4/q_emb_v4b.npy')
d_emb = np.load('/kaggle/input/notebooks/catalinalupu/notebook-varianta4/d_emb_v4b.npy')

full_id_to_idx = {mid: i for i, mid in enumerate(df_valid['movie_id'].tolist())}
eval_indices = [full_id_to_idx[mid] for mid in eval_df['movie_id'].tolist()]

test_q_emb = q_emb[eval_indices]

hits = {k: 0 for k in [1, 5, 10, 20]}
n = len(eval_df)

for i, gi in enumerate(eval_indices):
    sims = test_q_emb[i] @ d_emb.T
    rank = int((sims > sims[gi]).sum()) + 1
    for k in hits:
        if rank <= k:
            hits[k] += 1

print('V4b pe filme nevazute (eval 10%) vs pool complet 40k:')
for k in [1, 5, 10, 20]:
    print(f'  Hit@{k} = {hits[k]/n:.1%}')
print('\nComparatie: V4b pe toti 40k = 38.7%')

V4b pe filme nevazute (eval 10%) vs pool complet 40k:
  Hit@1 = 8.9%
  Hit@5 = 14.5%
  Hit@10 = 17.3%
  Hit@20 = 21.6%

Comparatie: V4b pe toti 40k = 38.7%


In [2]:
import numpy as np
import pandas as pd
import ast
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv')
for col in ['overview', 'overview_summary', 'review_summary', 'tagline', 'genres', 'keywords', 'cast']:
    df[col] = df[col].fillna('').astype(str).str.strip().replace('nan', '')

def clean_overview_summary(title, ovs):
    t = str(title).strip()
    s = str(ovs).strip()
    if len(t) > 3 and s.lower().startswith(t.lower()):
        s = s[len(t):].lstrip('. \n').strip()
    return s if s else str(ovs).strip()

def extract_cast_names(cast_raw, max_actors=4):
    if not isinstance(cast_raw, str) or not cast_raw.strip():
        return ''
    try:
        actors = ast.literal_eval(cast_raw)
        if isinstance(actors, list):
            return ', '.join(a['name'] for a in actors[:max_actors] if isinstance(a, dict) and a.get('name'))
    except:
        pass
    return ''

def build_doc_text_v4(row):
    parts = [str(row.get('title', '')).strip() + '.']
    ovs = clean_overview_summary(row.get('title', ''), row.get('overview_summary', ''))
    ov  = str(row.get('overview', '')).replace('nan', '').strip()
    plot = ovs if ovs else ov
    if plot: parts.append('Plot: ' + plot)
    rev = str(row.get('review_summary', '')).replace('nan', '').strip()
    if rev: parts.append('Critics: ' + rev)
    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres: parts.append('Genres: ' + genres)
    cast = extract_cast_names(row.get('cast', ''), 4)
    if cast: parts.append('Cast: ' + cast)
    return ' '.join(parts)

df['doc_text'] = df.apply(build_doc_text_v4, axis=1)
df_valid = df[(df['tagline'] != '') & (df['doc_text'] != '')].copy().reset_index(drop=True)

_, eval_df = train_test_split(df_valid, test_size=0.1, random_state=42)
eval_df = eval_df.reset_index(drop=True)

full_id_to_idx = {mid: i for i, mid in enumerate(df_valid['movie_id'].tolist())}
eval_indices = [full_id_to_idx[mid] for mid in eval_df['movie_id'].tolist()]

def eval_model(q_emb_path, d_emb_path, label, comparatie):
    q_emb = np.load(q_emb_path)
    d_emb = np.load(d_emb_path)
    test_q_emb = q_emb[eval_indices]
    hits = {k: 0 for k in [1, 5, 10, 20]}
    n = len(eval_df)
    for i, gi in enumerate(eval_indices):
        sims = test_q_emb[i] @ d_emb.T
        rank = int((sims > sims[gi]).sum()) + 1
        for k in hits:
            if rank <= k:
                hits[k] += 1
    print(f'\n{label} pe filme nevazute (eval 10%) vs pool complet 40k:')
    for k in [1, 5, 10, 20]:
        print(f'  Hit@{k} = {hits[k]/n:.1%}')
    print(f'Comparatie: {label} pe toti 40k = {comparatie}')

BASE = '/kaggle/input/notebooks/catalinalupu/notebook-varianta4/'

eval_model(BASE + 'q_emb_v4a.npy', BASE + 'd_emb_v4a.npy', 'V4a (MNR, fara HN)', '23.8%')
eval_model(BASE + 'q_emb_v4b.npy', BASE + 'd_emb_v4b.npy', 'V4b (MNR + HN cross-genre)', '38.7%')


V4a (MNR, fara HN) pe filme nevazute (eval 10%) vs pool complet 40k:
  Hit@1 = 10.3%
  Hit@5 = 17.2%
  Hit@10 = 20.5%
  Hit@20 = 25.2%
Comparatie: V4a (MNR, fara HN) pe toti 40k = 23.8%

V4b (MNR + HN cross-genre) pe filme nevazute (eval 10%) vs pool complet 40k:
  Hit@1 = 8.9%
  Hit@5 = 14.5%
  Hit@10 = 17.3%
  Hit@20 = 21.6%
Comparatie: V4b (MNR + HN cross-genre) pe toti 40k = 38.7%
